# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Using cookies from /home/kfish/.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /home/kfish/Documents/Programming/Python/cs521fml/CCS521FA26HW/hw3/q3/linf_base.pt
100%|█████████████████████████████████████| 11.2M/11.2M [00:00<00:00, 20.3MB/s]
Using cookies from /home/kfish/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=74c79cdd-31ed-4f70-a0b7-073014a23f1c
To: /home/kfish/Documents/Programming/Python/cs521fml/CCS521FA26HW/hw3/q3/cifar10_train_10k.pt
100%|█████████████████████████████████████| 30.8M/30.8M [00:01<00:00, 28.2MB/s]
Using cookies from /home/kfish/.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /home/kfish/Documents/Programming/Python/cs521fml/CCS521FA26HW/hw3/q3/cifar10_tes

In [2]:
import time
from collections.abc import Callable, Iterable

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
print("device:", device)

NORMS = ("linf",)
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: xpu


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(
        32, device=x.device
    ).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(
        32, device=x.device
    ).view(1, 1, 32)
    return (
        padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j]
        .permute(0, 3, 1, 2)
        .contiguous()
    )


def batches(x, y, batch_size, shuffle):
    idx = (
        torch.randperm(len(x), device=x.device)
        if shuffle
        else torch.arange(len(x), device=x.device)
    )
    for k in range(0, len(x), batch_size):
        j = idx[k : k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer(
            "mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1)
        )
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [
            (1, 1),
            (1, 1),
            (2, 2),
            (2, 1),
            (4, 2),
            (4, 1),
            (8, 2),
            (8, 1),
        ]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta) -> torch.Tensor:
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y) -> torch.Tensor:
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True) -> torch.Tensor:
    delta = torch.zeros_like(x)
    if rand_init:
        delta.uniform_(-eps, eps)
    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        eta = alpha * torch.sign(grad)
        delta = clip_box(x, torch.clamp(delta + eta, -eps, eps))

    return x + delta

In [6]:
ATTACKS: dict[str, Callable[..., torch.Tensor]] = {
    "linf": pgd_linf,
}


def attack(model, x, y, norm, eps, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = eps / steps**0.5
    return ATTACKS[norm](model, x, y, eps, alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(
    model,
    steps=50,
    restarts=2,
    batch_size=250,
    eps_range: Iterable[float] = np.linspace(1 / 255, 9 / 255, num=9),
):
    model.eval()
    torch.manual_seed(
        0
    )  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *(f"{eps:.2}" for eps in eps_range))}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i : i + batch_size], y_test[i : i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        for eps in eps_range:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, "linf", eps, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[f"{eps:.2}"] += robust.sum().item()
        correct["clean"] += clean.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print(
        "  ".join(f"{k} {v:.1f}" for k, v in acc.items())
        + f"   ({time.time() - start:.0f} s)"
    )
    return acc

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def normal_train(model, epochs=10, lr=0.01, batch_size=128, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, epochs * (len(x_train) // batch_size)
    )
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(
            f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)"
        )
    model.eval()
    return model

In [9]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, epochs * (len(x_train) // batch_size)
    )
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(
            f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)"
        )
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [10]:
model_normal = normal_train(make_model(), epochs=5)
test_model(model_normal)

epoch 1: loss 1.882 (3 s)
epoch 2: loss 1.559 (5 s)
epoch 3: loss 1.377 (7 s)
epoch 4: loss 1.240 (8 s)
epoch 5: loss 1.157 (10 s)
clean 56.0  0.0039 37.9  0.0078 22.5  0.012 12.4  0.016 6.0  0.02 2.6  0.024 1.1  0.027 0.4  0.031 0.2  0.035 0.0   (103 s)


{'clean': 56.0,
 '0.0039': 37.9,
 '0.0078': 22.5,
 '0.012': 12.4,
 '0.016': 6.0,
 '0.02': 2.6,
 '0.024': 1.1,
 '0.027': 0.4,
 '0.031': 0.2,
 '0.035': 0.0}

In [11]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    adv_x = attack(model, x, y, norm, eps=8 / 255)
    model.train()
    return F.cross_entropy(model(adv_x), y)

In [12]:
model_linf = adv_train(make_model(), single_norm_loss, norm="linf", epochs=5)
test_model(model_linf)

epoch 1: loss 2.197 (12 s)
epoch 2: loss 2.093 (25 s)
epoch 3: loss 2.052 (37 s)
epoch 4: loss 2.025 (48 s)
epoch 5: loss 2.013 (60 s)
clean 39.3  0.0039 37.5  0.0078 36.1  0.012 34.3  0.016 32.2  0.02 30.4  0.024 28.8  0.027 26.6  0.031 24.9  0.035 22.7   (96 s)


{'clean': 39.3,
 '0.0039': 37.5,
 '0.0078': 36.1,
 '0.012': 34.3,
 '0.016': 32.2,
 '0.02': 30.4,
 '0.024': 28.8,
 '0.027': 26.6,
 '0.031': 24.9,
 '0.035': 22.7}

In [13]:
test_model(model_normal, steps=1)
test_model(model_linf, steps=1)

clean 56.0  0.0039 42.0  0.0078 32.1  0.012 24.9  0.016 18.8  0.02 13.9  0.024 11.0  0.027 7.8  0.031 6.1  0.035 4.3   (3 s)
clean 39.3  0.0039 38.0  0.0078 37.0  0.012 35.8  0.016 34.3  0.02 32.6  0.024 31.3  0.027 30.1  0.031 29.4  0.035 27.9   (3 s)


{'clean': 39.3,
 '0.0039': 38.0,
 '0.0078': 37.0,
 '0.012': 35.8,
 '0.016': 34.3,
 '0.02': 32.6,
 '0.024': 31.3,
 '0.027': 30.1,
 '0.031': 29.4,
 '0.035': 27.9}